In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sb
from datetime import datetime
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, GridSearchCV, KFold
from sklearn.feature_selection import RFE
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, FunctionTransformer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, BaggingRegressor
from catboost import CatBoostRegressor
from statsmodels.tsa.arima.model import ARIMA
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from scipy.stats import uniform, randint
from sklearn.pipeline import Pipeline
from skopt import BayesSearchCV
from skopt.space import Real, Integer
import gc
import lightgbm as lgb
import optuna
import catboost
%matplotlib inline
print("Libraries imported")

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

### ^ These imports are becoming messy :^(

# Loading the data into variables cc_train, cc_test, cc_sample

- cc_train = /kaggle/input/widsdatathon2023/train_data.csv
- cc_test = /kaggle/input/widsdatathon2023/test_data.csv
- cc_sample = /kaggle/input/widsdatathon2023/sample_solution.csv

In [ ]:
cc_train = pd.read_csv('/kaggle/input/widsdatathon2023/train_data.csv')
cc_test = pd.read_csv('/kaggle/input/widsdatathon2023/test_data.csv')
cc_sample = pd.read_csv('/kaggle/input/widsdatathon2023/sample_solution.csv')

# Reducing the memory usage of the dataset

## Acknowledgement

[Reduce Dataframe size](https://www.kaggle.com/competitions/widsdatathon2023/discussion/376649)

In [ ]:
def reduce_mem_usage(dataframe, verbose=True):
  numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
  start_memory = dataframe.memory_usage().sum() / 1024**2
  for col in dataframe.columns:
    col_type = dataframe[col].dtypes
    if col_type in numerics:
      c_min = dataframe[col].min()
      c_max = dataframe[col].max()
      if str(col_type)[:3] == 'int':
        if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
          dataframe[col] = dataframe[col].astype(np.int8)
        elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
          dataframe[col] = dataframe[col].astype(np.int16)
        elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
          dataframe[col] = dataframe[col].astype(np.int32)
        elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
          dataframe[col] = dataframe[col].astype(np.int64)
      else:
        if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
          dataframe[col] = dataframe[col].astype(np.float32)
        else:
          dataframe[col] = dataframe[col].astype(np.float64)
  end_memory = dataframe.memory_usage().sum() / 1024**2
  print('Mem. usage decreased to {:5.2f} Mb ({:.1f}% reduction)'.format(end_memory, 100 * (start_memory - end_memory) / start_memory)) if verbose else print('Reduced to {:5.2f}'.format(end_memory))
  return dataframe

In [ ]:
cc_train = reduce_mem_usage(cc_train)
cc_test = reduce_mem_usage(cc_test)

## Determining variance in the dataset

In [ ]:
def evaluate_variance(df, threshold=0.01):
    variances = df.var()
    variance_dict = {}
    for col, var in variances.items():
        if var < threshold:
            variance_dict[col] = "low variance"
        else:
            variance_dict[col] = "high variance"
    return variance_dict

In [ ]:
#column__with_low_or_high_variance = evaluate_variance(cc_train.iloc[:10])

In [ ]:
'''
for key, val in column__with_low_or_high_variance.items():
    if 'wind' in key:
        print(f'{key}: {val}')
'''

# Overview of the dataset

Using shape, info, head, describe

# Tukey's Method to detect outliers

---

Tukey's method is used for identifying and removing outliers from a dataset. It involves using the interquartile range (IQR), which is the range between the 25th and 75th percentiles of the data, to determine whether a data point is an outlier. Any data points that fall outside of the range (Q1 - 1.5IQR, Q3 + 1.5IQR) are considered outliers and can be removed. This method is a simple and effective way to handle outliers in a dataset.

In [ ]:
from scipy.stats import iqr


def remove_outliers_tukey(data, alpha=1.5):
    '''
    Remove outliers using Tukey's method with the interquartile range (IQR).
    
    Parameters:
    data (numpy array or pandas dataframe): The data to remove outliers from.
    alpha (float): The sensitivity parameter, which determines the range to consider outliers.
                   A value of 1.5 is the default, which is a commonly used value.
    
    Returns:
    numpy array or pandas dataframe: The data with outliers removed.
    '''
    # Select only the numerical columns
    num_cols = data.select_dtypes(include=[np.number]).columns
    data_num = data[num_cols]
    
    # Compute the first and third quartiles
    q1, q3 = np.percentile(data_num, [25, 75])
    
    # Compute the interquartile range (IQR)
    iqr_val = iqr(data_num)
    
    # Compute the range outside of which data points are considered outliers
    outlier_range = (q1 - alpha * iqr_val, q3 + alpha * iqr_val)
    
    # Identify the outliers and remove them
    outliers = (data_num < outlier_range[0]) | (data_num > outlier_range[1])
    data_num_no_outliers = data_num[~outliers]
    
    # Merge the numerical columns back into the original data frame
    data_no_outliers = pd.concat([data_num_no_outliers, data.select_dtypes(exclude=[np.number])], axis=1)
    
    return data_no_outliers





In [ ]:
#cc_train = remove_outliers_tukey(cc_train)

In [ ]:
#cc_test= remove_outliers_tukey(cc_test)

In [ ]:
#data_with_removed_outliers.describe()

In [ ]:
#data_with_removed_outliers.head(5)

# Sensitivity Analysis

**Error from function**

---

/opt/conda/lib/python3.7/site-packages/ipykernel_launcher.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

In [ ]:
from scipy.stats import ttest_ind, levene, shapiro


def perform_sensitivity_analysis(dataframe, target_column, alpha=0.05, sample_size=5000):
    # Identify numeric columns
    numeric_columns = dataframe.select_dtypes(include=[np.number]).columns.tolist()

    # Exclude the target column
    numeric_columns.remove(target_column)

    # Exclude categorical columns
    for col in dataframe.select_dtypes(include=['category']).columns:
        if col in numeric_columns:
            numeric_columns.remove(col)

    # Sample the data to speed up hypothesis testing
    if len(dataframe) > sample_size:
        dataframe = dataframe.sample(sample_size)
    # Initialize results list
    results = []

    # Loop through each numeric column
    for col in numeric_columns:
        # Create copies of the dataframe with and without outliers
        with_outliers = dataframe.copy()
        without_outliers = dataframe.copy()

        # Calculate the upper and lower bounds of the data without outliers
        q1, q3 = np.percentile(without_outliers[col], [25, 75])
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        # Replace values outside of the bounds with NaN
        without_outliers[col][(without_outliers[col] < lower_bound) | (without_outliers[col] > upper_bound)] = np.nan

        # Drop rows with NaN
        without_outliers = without_outliers.dropna()

        # Perform statistical tests on the two sets of data
        ttest_p = ttest_ind(with_outliers[target_column], without_outliers[target_column], equal_var=False)[1]
        levene_p = levene(with_outliers[target_column], without_outliers[target_column])[1]
        shapiro_p = shapiro(without_outliers[target_column])[1]

        # Check if the results are statistically significant
        ttest_significant = ttest_p < alpha
        levene_significant = levene_p < alpha
        shapiro_significant = shapiro_p < alpha

        # Add the results to the list
        results.append({
            'column': col,
            'outliers_removed': len(with_outliers) - len(without_outliers),
            'ttest_p': ttest_p,
            'ttest_significant': ttest_significant,
            'levene_p': levene_p,
            'levene_significant': levene_significant,
            'shapiro_p': shapiro_p,
            'shapiro_significant': shapiro_significant
        })

    # Convert the results to a dataframe and return it
    return pd.DataFrame(results)

In [ ]:
#df_train_temp = cc_train.copy()

In [ ]:
#results = perform_sensitivity_analysis(cc_train, 'contest-tmp2m-14d__tmp2m')

In [ ]:
#print(results)

## Visualise the time gap between the train and the test data

In [ ]:
'''
# create a scatterplot using seaborn and matplotlib
fig, ax = plt.subplots(figsize=(16, 8))
sb.scatterplot(x=train_df.startdate, y=1, color='blue', label='Train Data', ax=ax)
sb.scatterplot(x=test_df.startdate, y=1, color='red', label='Test Data', ax=ax)

# add labels and a title to the plot
ax.set_title('Scatterplot of Startdate and Target')
ax.set_xlabel('Startdate')
ax.set_ylabel('Target')
ax.legend(loc='upper left')

# display the plot
plt.show()
'''


**Based on the graph above the train data was from 2014 - 2016 and the test data is from 2022. There is a big time gap between the train and test data.**

## Analyising the distribution of values in both the train and test dataset

In [ ]:
train_df = cc_train.copy()
test_df = cc_test.copy()

In [ ]:
def dist_of_train_test_features(train_df, test_df, feature):
    # Analyse the distribution of values in the training dataset
    plt.figure(figsize=(16, 5))
    sb.displot(data=train_df, x=feature, kind='kde', label='Train Data')

    # Analyse the distribution of values in the testing dataset
    plt.figure(figsize=(16, 5))
    sb.displot(data=test_df, x=feature, kind='kde', label='Test Data')
dist_of_train_test_features(train_df, test_df, 'nmme0-tmp2m-34w__nmme0mean')

In [ ]:
cc_train.shape

In [ ]:
cc_train.describe()

In [ ]:
cc_test.shape

In [ ]:
cc_sample.describe()

# Visualizing the target variable
`contest-tmp2m-14d__tmp2m`

In [ ]:
def target_var_visualized():
  plt.figure(figsize=(15,7))
  plt.subplot(121)
  sb.kdeplot(cc_train['contest-tmp2m-14d__tmp2m'], color = "#ffd514")
  plt.subplot(122)
  sb.boxplot(data=cc_train['contest-tmp2m-14d__tmp2m'], color = "#ff355d")
target_var_visualized()

## Displaying the distribution of the target variable in the train

In [ ]:
def histogram_plot(data, label, title):
    sb.histplot(data, color='blue', label=label)
    plt.legend()
    plt.title(title)
    plt.show()
histogram_plot(data=cc_train['contest-tmp2m-14d__tmp2m'], label="contest-tmp2m-14d__tmp2m", title="Target Variable distribution")

**The target variable is not in the test dataset**

### Simple for loop to list out the features from the train dataset

In [ ]:
'''
test_col = cc_test.columns
count = 0
for name in test_col:
  if count % 50 == 0:
    print()
  print(name + ', ', end='')
  count += 1
'''

In [ ]:
cc_train.columns[cc_train.isna().any()].tolist()

In [ ]:
cc_test.columns[cc_test.isna().any()].tolist()

In [ ]:
cc_sample.columns[cc_sample.isna().any()].tolist()

In [ ]:
def train_test_dist(train, test):
    fig, ax = plt.subplots(figsize = (10, 5))
    sb.kdeplot(data=train,  color='blue', fill=True, ax=ax, label="Train Data")
    sb.kdeplot(data=test, color='orange', fill=True, ax=ax, label="Test Data")
    plt.legend()
    plt.show()
    
#train_target = cc_train['contest-tmp2m-14d__tmp2m']
#test_target = cc_test['contest-tmp2m-14d__tmp2m']
target_to_plot = 'mei__meirank'
train_target = cc_train[target_to_plot]
test_target = cc_test[target_to_plot]
train_test_dist(train_target, test_target)

# Visualising Categorical columns

In [ ]:

# Plot bar plots for all categorical columns
for column in cc_train.select_dtypes(include=['object']).columns:
    cc_train[column].value_counts().plot(kind='bar', figsize=(10,5))
    plt.title(column)
    plt.show()
    

In [ ]:
'''
def plot_boxplots(data, target_column):
    num_cols = data.select_dtypes(exclude=['object']).columns
    for col in num_cols:
        plt.figure(figsize=(10, 5))
        sb.boxplot(x=target_column, y=col, data=data)
        plt.title(col + " vs " + target_column)
        plt.show()
plot_boxplots(cc_train, 'contest-tmp2m-14d__tmp2m')
'''

# Visualising Numerical Columns

In [ ]:
# Plot histograms for all numerical columns
num_cols = cc_train.columns
def plot_histograms(data, column_list):
    """
    This function plots histograms based on the number of columns and data provided.
    
    Parameters
    ----------
    data : pandas dataframe
        The data to be plotted.
    column_list : a list
        The list of 10 numerical column names from the dataset..
    
    Returns
    -------
    void
        Displays histograms from matplotlib.
        The histograms that are being plotted are frequency histograms 
        for the numerical columns specified in the column_list parameter .
    """
    num_cols = column_list
    for col in num_cols:
        plt.figure(figsize=(8, 6))
        data[col].hist(bins=50)
        plt.title(col)
        plt.tight_layout()
        plt.show()
        
# plot_histograms(cc_train, num_cols[:10])

# Feature engineering Season

In [ ]:
def add_season(df):
    season_map = [0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 0]
    df['season'] = df['month'].map(lambda x: season_map[x-1])

In [ ]:
def sin_transformer(period):
    """
    Returns a FunctionTransformer that applies the sine transformation to a cyclical feature.

    Parameters:
        period (int or float): The period of the cyclical feature, which is the number of units in one complete cycle.
        
    Returns:
        FunctionTransformer: A FunctionTransformer that applies the sine transformation to a cyclical feature.
    """
    return FunctionTransformer(lambda x: np.sin(x / period * 2 * np.pi))


def cos_transformer(period):
    """
    Returns a FunctionTransformer that applies the cosine transformation to a cyclical feature.

    Parameters:
        period (int or float): The period of the cyclical feature, which is the number of units in one complete cycle.
        
    Returns:
        FunctionTransformer: A FunctionTransformer that applies the cosine transformation to a cyclical feature.
    """
    return FunctionTransformer(lambda x: np.cos(x / period * 2 * np.pi))

## Definition of `encode_cyclic_features()` function

---

Function Name: encode_cyclic_features

Input:

- df: Pandas DataFrame containing columns 'dayofyear', 'month', and 'season'

Output:

- Pandas DataFrame with additional columns for encoded cyclical features and the original columns dropped
Functionality:

- Encodes cyclical features of day of year, month, and season using sine and cosine transformations
- Returns the updated DataFrame

Example Usage:

- `new_df = encode_cyclic_features(df)`

In [ ]:
def encode_cyclic_features(df):

    df['day_sin'] = sin_transformer(365).fit_transform(df['day'])
    df['day_cos'] = cos_transformer(365).fit_transform(df['day'])
    
    df['month_sin'] = sin_transformer(12).fit_transform(df['month'])
    df['month_cos'] = cos_transformer(12).fit_transform(df['month'])
    
    df['season_sin'] = sin_transformer(4).fit_transform(df['season'])
    df['season_cos'] = cos_transformer(4).fit_transform(df['season'])
    
    
    return df

# Data Preproccessing

In [ ]:
def location_feature(train, test):
    # Reference: https://www.kaggle.com/code/flaviafelicioni/wids-2023-different-locations-train-test-solved
    scale = 14
    train.loc[:,'lat']=round(train.lat,scale)
    train.loc[:,'lon']=round(train.lon,scale)
    test.loc[:,'lat']=round(test.lat,scale)
    test.loc[:,'lon']=round(test.lon,scale)
    
    train_and_test = pd.concat([train, test], axis=0)
    train_and_test['loc_group'] = train_and_test.groupby(['lat', 'lon']).ngroup()
    print(f'{train_and_test.loc_group.nunique()} unique locations')
    
    train = train_and_test.iloc[:len(train)]
    test = train_and_test.iloc[len(train):].drop(target, axis=1)
    
    return train, test

def cat_encode(train, test):
    """
    Encode the categorical feature in the train and test data set using LabelEncoder.

    Args:
        train (pandas.DataFrame): The training dataset.
        test (pandas.DataFrame): The testing dataset.

    Returns:
        pandas.DataFrame: The encoded training dataset.
        pandas.DataFrame: The encoded testing dataset.
    """
    le = LabelEncoder()
    train['climateregions__climateregion'] = le.fit_transform(train['climateregions__climateregion'])
    test['climateregions__climateregion'] = le.transform(test['climateregions__climateregion'])
    
    return train, test


def fill_na_rows(dataset):
    """
    Find the columns with missing values in the dataset and impute the missing values with the mean value of that column.

    Args:
        dataset (pandas.DataFrame): The dataset to fill missing values in.

    Returns:
        pandas.DataFrame: The dataset with missing values filled with the mean value of that column.
    """
    columns_with_missing_values = dataset.columns[dataset.isnull().any()].tolist()
    
    for col in columns_with_missing_values:
        dataset[col].fillna(dataset[col].mean(), inplace=True)
        
    return dataset


def create_new_feat(dataset):
    """
    Create new features year, month, and day from the startdate column of the dataset.

    Args:
        dataset (pandas.DataFrame): The dataset to create new features for.

    Returns:
        pandas.DataFrame: The dataset with new features year, month, and day.
    """
    dataset['year'] = pd.DatetimeIndex(dataset['startdate']).year 
    dataset['month'] = pd.DatetimeIndex(dataset['startdate']).month 
    dataset['day'] = pd.DatetimeIndex(dataset['startdate']).day
    return dataset


def feature_engineering(origin_train, origin_test):
    """
    Perform feature engineering on the training and testing datasets.

    Args:
        origin_train (pandas.DataFrame): The original training dataset.
        origin_test (pandas.DataFrame): The original testing dataset.

    Returns:
        pandas.DataFrame: The training features.
        pandas.DataFrame: The training targets.
        pandas.DataFrame: The testing features.
    """
    train, test = origin_train, origin_test
    train, test = location_feature(train, test)
    train = fill_na_rows(train)
    train = create_new_feat(train)
    test = create_new_feat(test)
    train, test = cat_encode(train, test)
    irrelevant_cols = ['index', 'startdate', 'contest-tmp2m-14d__tmp2m', 'climateregions__climateregion']
    features = [col for col in train.columns if col not in irrelevant_cols]
    X = train[features]
    X_test = test[features]
    y = train['contest-tmp2m-14d__tmp2m']

    return X, y, X_test


    
    

# Machine Learning Models

The goal of machine learning is to design intelligent systems. These systems can improve their performance over time without being explicitly programmed. The focus is on algorithms and models that can learn patterns and relationships in data. 

There are problems that occur when the model learn patterns or relationships in the data. These problems are overfitting and underfitting. 

Overfitting is when a model is too complex and captures noise in the training data rather than underlying patterns.

Underfitting is when a model is too simple and is unable to capture the underlying pattern.

The goal is to find a model that performs well on new, unseen data. This requires finding a balance between complexity and generalization. 

**Low Variance and Low Bias**

---

## RandomForestRegressor

Random Forest Regressor is a commonly used machine learning algorithm for regression problems and was chosen in this case because:

- It can handle both linear and non-linear relationships between features and target variables.

- It can handle missing data and is robust to noisy data.

- It is an ensemble method, which means it combines multiple decision trees to produce a more accurate and stable prediction.

- It can provide feature importance scores, which can be useful in identifying the most important features in the data.

- It is easy to implement and provides good results out-of-the-box, especially for large datasets with a large number of features.

These properties make Random Forest Regressor a good choice for a first attempt at solving this regression problem, and it can be a good starting point for further tuning and optimization

# Splittin the data set to train the model

Target variable: `contest-tmp2m-14d__tmp2m`

[WiDS Datathon Challenge](https://www.kaggle.com/competitions/widsdatathon2023/data)

Definition of target variable: 
- the arithmetic mean of the max and min observed temperature over the next 14 days for each location and start date, is provided

Evaluation Metric:

[Evaluation Reference](https://www.kaggle.com/competitions/widsdatathon2023/overview/evaluation)

Root Mean Squared Error (RMSE)

In [ ]:
target="contest-tmp2m-14d__tmp2m"
cc_test_copy = cc_test.copy()
cc_test_copy_v3 = cc_test.copy()

In [ ]:
# Split the data into training and test sets
X, y, X_test = feature_engineering(cc_train.copy(), cc_test.copy())



In [ ]:
x_train_with_seasonal_feat = X.copy()
x_test_with_seasonal_feat = X_test.copy()

In [ ]:
add_season(x_train_with_seasonal_feat)
add_season(x_test_with_seasonal_feat)

In [ ]:
x_train_with_seasonal_feat = encode_cyclic_features(x_train_with_seasonal_feat)
x_test_with_seasonal_feat = encode_cyclic_features(x_test_with_seasonal_feat)

### Visualizing the data after feature engineering

In [ ]:
target_to_plot = 'contest-pevpr-sfc-gauss-14d__pevpr'
train_target = X[target_to_plot]
test_target = X_test[target_to_plot]
train_test_dist(train_target, test_target)

## Conducting PCA on a sample of 10000

In [ ]:
sample_df = X.sample(n=10000, random_state=42)
# Scaling the data
scaler = StandardScaler()
scaled_data = scaler.fit_transform(X)

# Perform PCA
pca = PCA(n_components=10)
pca_data = pca.fit_transform(scaled_data)

# Get the explained variance ratio of each principal component
explained_variances = pca.explained_variance_ratio_


# Get the relevant features for each principal component
feature_names = X.columns
components = pd.DataFrame(pca.components_, columns=feature_names)

# Print the top relevant features for each principal component
for i in range(len(explained_variances)):
    print(f"Principal Component {i+1}:")
    print(components.iloc[i].sort_values(ascending=False)[:20]) # top 20
    
'''
# Visualize the explained variance ratio
import matplotlib.pyplot as plt
plt.bar(range(1, len(explained_variances) + 1), explained_variances)
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.show()
'''

# Checking Correlation

In [ ]:
## Identify correlated features to drop that fall above a correlation threshold 
## https://goodboychan.github.io/python/datacamp/machine_learning/2020/07/08/02-Feature-selection-I-selecting-for-feature-information.html 

def identify_correlated(df, threshold):
    corr_matrix = df.corr().abs()
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    reduced_corr_matrix = corr_matrix.mask(mask)
    features_to_drop = [c for c in reduced_corr_matrix.columns if any(reduced_corr_matrix[c] > threshold)]
    return features_to_drop

In [ ]:
# afterwards i should print out the columns that are of high importance from the models
# perform PCA and see if it determines the same features as the corr_matrix
# at.96 the score is ~1.24 (~51 columns dropped)
# at .70 the score is ~1.4 (~100+ columns dropped)
# at .80 the score is ~0.968 (100 columns dropped) - but why were these columns so unimportant that dropping it had a better outcome as compared to the other trials with .95 and .70 ?
# **Besides the fact that they were highly correlated - at .70 had more columns but dropping the columns identifed at .70 had a worse score than .80
features_to_drop = identify_correlated(cc_train, .80)

In [ ]:
print(len(features_to_drop))
print(features_to_drop)

In [ ]:
remove_feature = ['index', 'contest-tmp2m-14d__tmp2m']
features_to_drop_v1 = [ele for ele in features_to_drop if ele not in remove_feature]
features_to_drop_v1

***Dataset with seasonal features reduced***

In [ ]:
cc_train_with_seasonal_feat = pd.DataFrame(x_train_with_seasonal_feat.drop(features_to_drop_v1, axis=1))
cc_test_with_seasonal_feat = pd.DataFrame(x_test_with_seasonal_feat.drop(features_to_drop_v1, axis=1))
print("Dropped features that are highly correlated in the new data set with seasonal features")

In [ ]:
cc_train_reduced = pd.DataFrame(X.drop(features_to_drop_v1, axis=1))
cc_test_reduced = pd.DataFrame(X_test.drop(features_to_drop_v1, axis=1))
print("Dropped features that are highly correlated")

def ensemble_predict(xgboost_preds, lightgbm_preds):
    # combine the predictions of the two models
    combined_preds = np.mean([xgboost_preds, lightgbm_preds], axis=0)
    return combined_preds

In [ ]:
X_train, X_test_tts, y_train, y_test = train_test_split(cc_train_reduced, y, test_size=0.33, random_state=42)
print("Split the dataset for training successfully")

In [ ]:
X_train_seasonal, X_test_tts_seasonal, y_train_seasonal, y_test_seasonal = train_test_split(cc_train_with_seasonal_feat, y, test_size=0.33, random_state=42)
print("Split the dataset for training successfully")

# Using RandomForestRegressor

**Training and test performance**

In [ ]:
'''
# Train the Random Forest Regressor
params = {
    'n_estimators': 5000,
    'max_depth': 10,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'max_features': 'sqrt',
    'bootstrap': True,
    'oob_score': True
}
regr_rfr = RandomForestRegressor(**params)
regr_rfr.fit(X_train, y_train)
# make predictions on the test data
y_pred_rfr = regr_rfr.predict(X_test_tts)

# calculate the RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred_rfr))
print("RMSE:", rmse)
'''

In [ ]:
# cc_test_pred = regr_rfr.predict(cc_test_reduced)

In [ ]:
# cc_test_copy[target] = cc_test_pred
# cc_test_copy[[target,"index"]].to_csv("rfrpredictions.csv",index = False)

## Currently the XGBoost and lightgbm models are being trained on a reduced train dataset

---

`features_to_drop = identify_correlated(cc_train, .80)` is the line of code that identifies the features with 80% correlation. These features are removed from the train and test dataset. The reduced datasets are then utilized in the two models.

# Using XGBoost

*Gradient Boosted Decision Trees*

XGBoost is short for Extreme Gradient Boosting. The implementation is designed for speed and performance. It is an efficient implementation of the stochastic gradient boosting algorithm.

### Hyperparameters used
These parameters are used to specify the hyperparameters for a gradient boosting tree (GBT) model in the XGBoost library.

- **base_score**: It is the initial prediction score of all instances, global bias.

- **booster**: It specifies which booster to use for model training. The value 'gbtree' indicates that a tree-based booster will be used.

- **tree_method**: It specifies the method used to build the trees. The value 'gpu_hist' means that the histogram-based algorithm is used to build the trees on a GPU.

- **n_estimators**: It is the number of trees in the forest. The higher the number, the more complex the model becomes, but also the longer it takes to train.

- **early_stopping_rounds**: It is used to stop the training process early when the performance on a validation set starts to degrade. The value 50 indicates that training will be stopped if the performance on the validation set does not improve after 50 iterations.

- **objective**: It defines the loss function to be minimized. The value 'reg:squarederror' means that the model will minimize the mean squared error between the predicted and actual values.

- **max_depth**: It is the maximum depth of the trees in the model. The higher the value, the more complex the model becomes. Note: 2 - 8 is recommended, any higher value than 8 would not provide any more benefits.

- **subsample**: It is the fraction of the training instances used to build each tree in the forest. The lower the value, the simpler the model becomes, but also the more prone to overfitting.

- **colsample_bytree**: It is the fraction of the columns used to build each tree in the forest. The lower the value, the simpler the model becomes, but also the more prone to overfitting.

- **learning_rate**: It is the step size at which the optimizer makes updates to the model weights. A lower value means that the model updates more slowly but with less noise.

- **gpu_id**: It is the GPU device id to use for training. The value 0 indicates that the first GPU in the system will be used.

*Can use StratifiedKfold and gridsearhcv to determine the best combinations of parameters.*

> Note 1: I would like to reduce the value of the n_estimators will maintaining the same rmse or better.

> Note 2: Review the learning curve on the training and validation set

In [ ]:
cc_sample_preds = cc_sample['contest-tmp2m-14d__tmp2m']

In [ ]:

# set up parameters for XGBoost
# list of learning_rates to test [0.0001, 0.001, 0.01, 0.1, 0.2, 0.3]
# n_estimators = [50, 100, 150, 200]
# max_depth = [2, 4, 6, 8]
# param_grid = dict(max_depth=max_depth, n_estimators=n_estimators)
print("Training and predicting using xgboost")
# Define the search space for the hyperparameters
params = {'base_score': 0.5, 
          'booster': 'gbtree',
          'n_estimators': 15000,
          'objective': 'reg:squarederror',
          'max_depth': 5,
          'subsample': 0.6192476575209984,
          'colsample_bytree': 0.789420916404835,
          'lambda': 10,
          'min_child_weight': 15,
          'learning_rate': 0.05013399117431618}

reg_xgb = xgb.XGBRegressor(**params)

'''
reg_xgb.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_test_tts, y_test)], verbose=1000)


# get the feature importance scores
importance_scores = reg_xgb.feature_importances_
feature_importances = pd.DataFrame({'feature': X_train.columns, 'importance': importance_scores})
feature_importances.to_csv("xgboostbestparameters.csv")
# sort the features by importance score
feature_importances = feature_importances.sort_values('importance', ascending=False)
print(feature_importances)



# make predictions on the test data
y_pred_xgb = reg_xgb.predict(X_test_tts)

# calculate the RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
print("RMSE:", rmse)


cc_test_pred = reg_xgb.predict(cc_test_with_seasonal_feat)
cc_test_copy[target] = cc_test_pred
cc_test_copy[[target,"index"]].to_csv("xgbpredictions.csv",index = False)
print("Finished training and fitting, created xgbpredictions,csv")
'''


## Checking the rmse score with respect to different parameters for xgboost

def get_rmse(max_depth, learning_rate, n_estimators, train_X, val_X, train_y, val_y):
    model = xgb.XGBRegressor(max_depth=max_depth, learning_rate=learning_rate, n_estimators=n_estimators)
    model.fit(train_X, train_y)
    preds_val = model.predict(val_X)
    rmse = np.sqrt(mean_squared_error(val_y, preds_val))
    return rmse

def get_rmse(max_depth, learning_rate, n_estimators, train_X, val_X, train_y, val_y):
    model = RandomForestRegressor(max_depth=max_depth, learning_rate=learning_rate, n_estimators=n_estimators)
    model.fit(train_X, train_y)
    preds_val = model.predict(val_X)
    rmse = np.sqrt(mean_squared_error(val_y, preds_val))
    return rmse

## Hyperparameter tuning slow way

for max_depth in [3, 6, 9]:
    for learning_rate in [0.1, 0.3, 0.5]:
        for n_estimators in [50, 100, 150]:
            my_rmse = get_rmse(max_depth, learning_rate, n_estimators, X_train, X_test_tts, y_train, y_test)
            print("Max depth: %d  \t Learning rate: %.1f  \t N estimators: %d  \t RMSE: %.2f" %(max_depth, learning_rate, n_estimators, my_rmse))

param_grid = {
    'max_depth': [3, 6, 9],
    'learning_rate': [0.1, 0.3, 0.5],
    'n_estimators': [50, 100, 150]
}

model = xgb.XGBRegressor()

grid_search = GridSearchCV(
    model, param_grid, cv=5, scoring='neg_root_mean_squared_error', verbose=1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best RMSE:", -grid_search.best_score_)

In [ ]:
def get_important_features(df, threshold):
    """
    Returns a list of feature names whose importance scores are greater than or equal to the threshold value
    
    Parameters:
        df (pandas.DataFrame): DataFrame containing the feature importance scores
        threshold (float): Importance score threshold
        
    Returns:
        list: List of feature names whose importance scores are greater than or equal to the threshold value
    """
    # Filter the DataFrame by the threshold value
    important_features = df[df['importance'] >= threshold]
    
    # Get the names of the important features
    important_feature_names = list(important_features['feature'])
    
    return important_feature_names

In [ ]:
#importance_scores = reg_xgb.feature_importances_
#feature_importances = pd.DataFrame({'feature': X_train.columns, 'importance': importance_scores})

In [ ]:
#feature_importances = feature_importances.sort_values('importance', ascending=False)

In [ ]:
#all_important_features_above_threshold = get_important_features(feature_importances, 0.0001)

In [ ]:
#print(len(all_important_features_above_threshold))

In [ ]:
#cc_train_impt_feats = cc_train_reduced[all_important_features_above_threshold[:130]]
#cc_test_impt_feats = cc_test_reduced[all_important_features_above_threshold[:130]]
#print("Using Important Features learned from XGBoost")

In [ ]:
#X_train_impt_feat, X_test_tts_impt_feat, y_train_impt_feat, y_test_impt_feat = train_test_split(cc_train_impt_feats, y, test_size=0.33, random_state=42)
#print("Split the dataset with new feats for training successfully")

In [ ]:
'''
print("Using XGBoost ")
# Define the search space for the hyperparameters
params = {'base_score': 0.5, 
          'booster': 'gbtree',
          'tree_method': 'gpu_hist',
          'n_estimators': 25000,
          'objective': 'reg:squarederror',
          'max_depth': 5,
          'subsample': 0.6192476575209984,
          'colsample_bytree': 0.789420916404835,
          'gamma': 0.44484790661447615,
          'min_child_weight': 15,
          'learning_rate': 0.05013399117431618,
          'gpu_id': 0}

reg_xgb = xgb.XGBRegressor(**params)

reg_xgb.fit(X_train_impt_feat, y_train_impt_feat, eval_set=[(X_train_impt_feat, y_train_impt_feat), (X_test_tts_impt_feat, y_test_impt_feat)], verbose=1000)


# get the feature importance scores
importance_scores = reg_xgb.feature_importances_
feature_importances = pd.DataFrame({'feature': X_train.columns, 'importance': importance_scores})
feature_importances.to_csv("xgboostbestparameters.csv")
# sort the features by importance score
feature_importances = feature_importances.sort_values('importance', ascending=False)
print(feature_importances)



# make predictions on the test data
y_pred_xgb_v1 = reg_xgb.predict(X_test_tts_impt_feat)

# calculate the RMSE
rmse = np.sqrt(mean_squared_error(y_test_impt_feat, y_pred_xgb_v1))
print("RMSE:", rmse)


cc_test_pred_impt = reg_xgb.predict(cc_test_impt_feats)
cc_test_copy[target] = cc_test_pred_impt
cc_test_copy[[target,"index"]].to_csv("xgbpredictions.csv",index = False)
print("Finished training and fitting, created xgbpredictions,csv")
print(np.sqrt(mean_squared_error(cc_sample_preds, cc_test_pred_impt)))
'''

In [ ]:
'''
def objective(trial):
    params = {
        'base_score': 0.5, 
        'booster': 'gbtree',
        'tree_method': 'gpu_hist',
        'n_estimators': 10000,
        'objective': 'reg:squarederror',
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_uniform('subsample', 0.1, 1.0),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.1, 1.0),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.001, 0.1),
        'gpu_id': 0
    }

    reg_xgb = xgb.XGBRegressor(**params)

    reg_xgb.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_test_tts, y_test)], verbose=1000)

    y_pred_xgb = reg_xgb.predict(X_test_tts)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred_xgb))

    return rmse

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=25)

print('Number of finished trials:', len(study.trials))
print('Best trial:')
trial = study.best_trial

print(f'  Value: {trial.value:.5f}')
print('  Params: ')
for key, value in trial.params.items():
    print(f'    {key}: {value}')

best_params = trial.params
reg_xgb = xgb.XGBRegressor(**best_params)
reg_xgb.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_test_tts, y_test)])
y_pred_xgb = reg_xgb.predict(X_test_tts)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
print("RMSE:", rmse)

cc_test_pred = reg_xgb.predict(cc_test_reduced)
cc_test_copy[target] = cc_test_pred
cc_test_copy[[target,"index"]].to_csv("xgbpredictions.csv",index = False)
print("Finished training and fitting, created xgbpredictions.csv")
'''

# XGBoost Model Evaluation

---

The measure of model quality is predictive accuracy. Will the model predictions be close to the actual values.

Let's summarize the models quality in an understandble way.

---

One metric we can use to summarize model quality is mean absolute error.

Prediction error can be calculated `error = actual - predicted`

With the mean absolute error, we take the absolute value of each error. We then take the average of those absolute values.

In [ ]:
#mean_absolute_error(y_test, y_pred_xgb)

# How does catboost manage

In [ ]:
from catboost import CatBoostRegressor

params = {'iterations': 15000,
          'learning_rate': 0.0884621450893729,
          'depth': 10,
          'l2_leaf_reg': 0.011792972850764019,
          'random_seed': 42,
          'bagging_temperature': 0.09154463270628772,
          'border_count': 120,
          'loss_function': 'RMSE',
          'verbose': 1000
         }

reg_cat = CatBoostRegressor(**params)
'''
reg_cat.fit(X_train, y_train, eval_set=[(X_test_tts, y_test)])


# get the feature importance scores
importance_scores = reg_cat.get_feature_importance()
feature_importances = pd.DataFrame({'feature': X_train.columns, 'importance': importance_scores})
feature_importances.to_csv("catboostbestparameters.csv")
# sort the features by importance score
feature_importances = feature_importances.sort_values('importance', ascending=False)
print(feature_importances)


# make predictions on the test data
y_pred_cat = reg_cat.predict(X_test_tts)

# calculate the RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred_cat))
print("RMSE:", rmse)

cc_test_copy_v5 = cc_test_copy.copy()
cc_test_pred_catb = reg_cat.predict(cc_test_reduced)
cc_test_copy_v5[target] = cc_test_pred_catb
cc_test_copy_v5[[target,"index"]].to_csv("catboostpredictions.csv",index = False)
print("Finished training and fitting, created catboostpredictions.csv")


cc_sample_preds = cc_sample['contest-tmp2m-14d__tmp2m']
check_rmse = np.sqrt(mean_squared_error(cc_sample_preds, cc_test_pred_catb))
print(f"RMSE: {check_rmse:.4f}")
'''


## Optuna CatBoost Hyperparameter Tuning

In [ ]:
'''
import optuna
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error

def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 1000, 15000),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.1),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'l2_leaf_reg': trial.suggest_loguniform('l2_leaf_reg', 0.01, 10.0),
        'model_size_reg': trial.suggest_loguniform('model_size_reg', 0.01, 10.0),
        'random_seed': 0,
        'loss_function': 'RMSE',
        'task_type': 'GPU',
        'verbose': 1000
    }
    
    reg_cb = CatBoostRegressor(**params)
    reg_cb.fit(X_train, y_train, eval_set=[(X_test_tts, y_test)], verbose=1000)
    y_pred_cb = reg_cb.predict(X_test_tts)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_cb))
    return rmse

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=25)

best_params = study.best_params
print("Best params: ", best_params)

reg_cb = CatBoostRegressor(**best_params)
reg_cb.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_test_tts, y_test)])

importance_scores = reg_cb.feature_importances_
feature_importances = pd.DataFrame({'feature': X_train.columns, 'importance': importance_scores})
feature_importances.to_csv("catboost_best_parameters.csv")
feature_importances = feature_importances.sort_values('importance', ascending=False)
print(feature_importances)

y_pred_cb = reg_cb.predict(X_test_tts)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_cb))
print("RMSE:", rmse)

cc_test_pred_cat = reg_cb.predict(cc_test_reduced)
cc_test_copy[target] = cc_test_pred_cat
cc_test_copy[[target,"index"]].to_csv("catboost_predictions.csv",index = False)
print("Finished training and fitting, created catboost_predictions.csv")
'''

In [ ]:
'''

from catboost import CatBoostRegressor
from bayes_opt import BayesianOptimization
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_absolute_error

# Define the function to optimize
def catboost_cv(
    iterations,
    learning_rate,
    subsample,
    l2_leaf_reg,
    max_depth,
    model_size_reg,
    X=X_train, y=y_train
):
    # Define the model with the given hyperparameters
    model = CatBoostRegressor(
        iterations=int(iterations),
        learning_rate=learning_rate,
        subsample=subsample,
        l2_leaf_reg=l2_leaf_reg,
        max_depth=int(max_depth),
        model_size_reg=model_size_reg,
        random_seed=42,
        silent=True
    )
    
    # Define cross-validation
    n_folds = 5
    folds = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    # Define evaluation metric
    metric = 'neg_mean_absolute_error'
    
    # Calculate cross-validation score
    scores = cross_val_score(model, X, y, cv=folds, scoring=metric, n_jobs=-1)
    
    # Return the mean cross-validation score
    return np.mean(scores)

# Define the hyperparameter search space
pbounds = {
    'iterations': (1000, 15000),
    'learning_rate': (0.001, 1.0),
    'subsample': (0.1, 1.0),
    'l2_leaf_reg': (0.01, 10.0),
    'max_depth': (3, 10),
    'model_size_reg': (0.01, 10.0)
}

# Define the Bayesian optimization object and run the optimization
catboost_bo = BayesianOptimization(f=catboost_cv, pbounds=pbounds, random_state=42)
catboost_bo.maximize(init_points=10, n_iter=40, acq='ucb', kappa=2.576)

# Print the best hyperparameters and the corresponding score
print(catboost_bo.max)
'''


# Using lasso and gradientboosting ensemble

This is taking too long. I need to find a way to speed up

# Using Lasso

lasso = Lasso(alpha=0.005, random_state=1, max_iter=1000)
lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test_tts)
cc_test_pred_lasso = lasso.predict(X_test)
cc_test_copy[target] = cc_test_pred_lasso
cc_test_copy[[target,"index"]].to_csv("lassopredictions.csv",index = False)

# Using GradientBoostingRegressor
gbr = GradientBoostingRegressor(n_estimators=10000, learning_rate=0.05, max_depth=4, max_features='sqrt', min_samples_leaf=15, min_samples_split=10, loss='huber', random_state =5)
gbr.fit(X_train, y_train)
y_pred_gbr = gbr.predict(X_test_tts)
cc_test_copy_v2 = cc_test_copy.copy()
cc_test_pred_gbr = gbr.predict(X_test)
cc_test_copy_v2[target] = cc_test_pred_gbr
cc_test_copy_v2[[target, "index"]].to_csv("gbrpredictions.csv", index = False)

# Using LightGBM

## understanding hyperparameters

[Amazon documentation on LightGBM](https://docs.aws.amazon.com/sagemaker/latest/dg/lightgbm-hyperparameters.html)

In [ ]:

print("Beginning training and fitting lightgbm model")
params = {
    'boosting_type': 'gbdt',
    'objective': 'regression',
    'metric': 'rmse',
    'num_leaves': 31,
    'max_depth': 8,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'n_estimators': 15000
}

# Create the LightGBM model object
reg_lgb = lgb.LGBMRegressor(**params)

'''
# Fit the model to the training data
reg_lgb.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_test_tts, y_test)], verbose=1000)

# Get feature importances
importance_scores = reg_lgb.feature_importances_
feature_importances = pd.DataFrame({'feature': X_train.columns, 'importance': importance_scores})

# Sort the features by importance score
feature_importances = feature_importances.sort_values('importance', ascending=False)

# Output feature importances to a CSV file
feature_importances.to_csv("lgbm_feature_importances.csv", index=False)


# Generate predictions on the test data
y_pred_lgb = reg_lgb.predict(X_test_tts)

# Calculate the RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred_lgb))
print("RMSE:", rmse)

# Make predictions on the competition test data
cc_test_copy_v3 = cc_test_copy.copy()
cc_test_pred_lgb = reg_lgb.predict(cc_test_reduced)
cc_test_copy_v3[target] = cc_test_pred_lgb
cc_test_copy_v3[[target, "index"]].to_csv("lgbpredictions.csv", index=False)
print("Finished training and fitting lightgbm model, created lgbpredictions.csv and feature_importances.csv")
'''


In [ ]:

'''

def objective(trial):
    params = {
        'boosting_type': 'gbdt', 
        'objective': 'regression', 
        'metric': 'rmse', 
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'num_leaves': trial.suggest_int('num_leaves', 10, 100),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.001, 0.1),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'device_type':'gpu',
        'n_estimators': 10000
    }

    reg_lgb = lgb.LGBMRegressor(**params)

    reg_lgb.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_test_tts, y_test)])

    y_pred_lgb = reg_lgb.predict(X_test_tts)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred_lgb))

    return rmse

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=15)

print('Number of finished trials:', len(study.trials))
print('Best trial:')
trial = study.best_trial

print(f'  Value: {trial.value:.5f}')
print('  Params: ')
for key, value in trial.params.items():
    print(f'    {key}: {value}')    

best_params = trial.params
reg_lgb = lgb.LGBMRegressor(**best_params)
reg_lgb.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_test_tts, y_test)])
y_pred_lgb = reg_lgb.predict(X_test_tts)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_lgb))
print("RMSE:", rmse)

cc_test_copy_v3 = cc_test_copy.copy()
cc_test_pred_lgb = reg_lgb.predict(cc_test_reduced)
cc_test_copy_v3[target] = cc_test_pred_lgb
cc_test_copy_v3[[target, "index"]].to_csv("lgbpredictions.csv", index=False)
'''


cc_test_copy_v2 = cc_test_copy.copy()
cc_test_pred_lgb = reg_lgb.predict(cc_test_reduced)
cc_test_copy_v2[target] = cc_test_pred_lgb
cc_test_copy_v2[[target,"index"]].to_csv("lgbpredictions.csv",index = False)

In [ ]:
def plot_last_predictions(y_true, y_pred_xgb, y_pred_lgb, y_pred_cat):
    last_n = 30
    plt.figure(figsize=(12, 6))
    plt.plot(y_true[-last_n:], marker='o', label='Actual')
    plt.plot(y_pred_xgb[-last_n:], marker='x', label='XGBoost')
    plt.plot(y_pred_lgb[-last_n:], marker='x', label='LightGBM')
    plt.plot(y_pred_lgb[-last_n:], marker='x', label='CatBoost')
    plt.legend()
    plt.title(f'Last {last_n} Predictions vs Actual')
    plt.show()

In [ ]:
#plot_predictions(cc_sample_preds, cc_test_pred, cc_test_pred_lgb, cc_test_pred_cat)

## Let's try out using Robust Linear regression model from the statsmodel api

---

Robust regression models can be particularly useful for large datasets with many features, where outliers are common and can significantly impact the performance of the model. By using robust regression models, you can improve the reliability of your predictions and reduce the risk of overfitting to noisy or spurious data.

In [ ]:
'''
import statsmodels.api as sm

# Fit a robust linear regression model
rlm_model = sm.RLM(y_train, X_train, M=sm.robust.norms.Hampel())
rlm_results = rlm_model.fit(scale_est=sm.robust.scale.HuberScale())

# Print the summary of the model
print(rlm_results.summary())

# Get the predicted values on the test set
y_pred = rlm_results.predict(X_test_tts)

# Calculate the RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("RMSE:", rmse)
'''

# Using Multi-Layer Perceptron Models

In [ ]:
'''
params = {'hidden_layer_sizes': (100,50,10), 
          'activation': 'relu',
          'solver': 'adam',
          'alpha': 0.0001,
          'batch_size': 'auto',
          'learning_rate': 'constant',
          'learning_rate_init': 0.001,
          'max_iter': 200,
          'shuffle': True,
          'random_state': None,
          'tol': 0.0001,
          'verbose': True,
          'warm_start': False,
          'momentum': 0.9,
          'nesterovs_momentum': True,
          'early_stopping': False,
          'validation_fraction': 0.1,
          'beta_1': 0.9,
          'beta_2': 0.999,
          'epsilon': 1e-08}

# Standardize the data using a standard scaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test_tts = scaler.transform(X_test_tts)

reg_mlp = MLPRegressor(**params)

reg_mlp.fit(X_train, y_train)

# make predictions on the test data
y_pred_mlp = reg_mlp.predict(X_test_tts)

# calculate the RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred_mlp))
print("RMSE:", rmse)
# Make predictions on the competition test data
cc_test_copy_v2 = cc_test_copy.copy()

cc_test_pred_mlp = reg_mlp.predict(cc_test_reduced)
cc_test_copy_v2[target] = cc_test_pred_mlp
cc_test_copy_v2[[target,"index"]].to_csv("mlppredictions.csv",index = False)
'''

cc_test_copy_v2[target]

cc_test_copy[target]

## Boosting technique for ensembling

In [ ]:
# Create a list of the three models
models = [reg_xgb, reg_cat, reg_lgb]

# Initialize the BaggingRegressor object
reg_bag = BaggingRegressor(base_estimator=reg_xgb,
                           n_estimators=5,
                           max_samples=0.5,
                           max_features=0.5)

# Fit the BaggingRegressor on the training data
reg_bag.fit(X_train, y_train)

# Make predictions using the stacking regressor
y_pred_ensemble_xgb = reg_bag.predict(X_test_tts)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_ensemble_xgb))
print("RMSE:", rmse)
rmse_cc_sample = np.sqrt(mean_squared_error(cc_sample_preds, y_pred_ensemble_xgb))
print('RMSE:', rmse_cc_sample)


In [ ]:
# Initialize the BaggingRegressor object
reg_bag = BaggingRegressor(base_estimator=reg_cat,
                           n_estimators=5,
                           max_samples=0.5,
                           max_features=0.5)

# Fit the BaggingRegressor on the training data
reg_bag.fit(X_train, y_train)

# Make predictions using the stacking regressor
y_pred_ensemble_cat = reg_bag.predict(X_test_tts)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_ensemble_cat))
print("RMSE:", rmse)
rmse_cc_sample = np.sqrt(mean_squared_error(cc_sample_preds, y_pred_ensemble_cat))
print('RMSE:', rmse_cc_sample)

In [ ]:
# Initialize the BaggingRegressor object
reg_bag = BaggingRegressor(base_estimator=reg_lgb,
                           n_estimators=5,
                           max_samples=0.5,
                           max_features=0.5)

# Fit the BaggingRegressor on the training data
reg_bag.fit(X_train, y_train)

# Make predictions using the stacking regressor
y_pred_ensemble_lgb = reg_bag.predict(X_test_tts)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_ensemble_lgb))
print("RMSE:", rmse)
rmse_cc_sample = np.sqrt(mean_squared_error(cc_sample_preds, y_pred_ensemble_lgb))
print('RMSE:', rmse_cc_sample)

In [ ]:
ensemble_preds = y_pred_ensemble_xgb*0.7+y_pred_ensemble_lgb*0.1+y_pred_ensemble_cat*0.2

In [ ]:
#ensemble_preds

In [ ]:
#ensemble_pred_mean = np.mean([cc_test_pred_lgb, cc_test_pred, cc_test_pred_cat], axis=0)

In [ ]:
#ensemble_pred_mean

In [ ]:

cc_submission = cc_test_copy.copy()
cc_submission[target] = ensemble_preds
cc_submission[[target,"index"]].to_csv('submission.csv', index = False)
